# Which strategies run in which space


**What this answers (instruction 53, WP8).** ParaScape has 39 strategies, written against the Toxoplasma space. Which of them can run in another space at all?

**How.** Each strategy declares what it needs -- "a categorical column", "two or more edge layers", "the other organism's table" -- and those words are checked against what each space actually holds. A strategy that cannot run names the requirement it is missing.

**What this is not.** A measurement of how well a strategy works in a space. That is the track record, and a space with no track record is never called calibrated.

Built 2026-11-09.

In [1]:
import json, os, numpy as np, pandas as pd
import parascape.strategy_graph, parascape.strategy_catalog, parascape.strategy_learning  # register them
from parascape import strategies, strategy_applicability as SA
SPACES = (('Tg', 'parascape/data/nodes.parquet', 'parascape/data/graph.npz'),
          ('Pf', 'parascape/data/pf_nodes.parquet', 'parascape/data/pf_graph.npz'),
          ('Cp', 'results/cryptosporidium_space_2026_11_08/cp_gene_nodes.parquet', None))
def layer_names(relative):
    if not relative:
        return ()
    path = os.path.join(REPO, relative)
    if not os.path.exists(path):
        return ()
    return tuple(np.load(path, allow_pickle=True).files)
print(len(strategies.REGISTRY), 'strategies registered')
rows = []
for code, table, graph in SPACES:
    nodes = pd.read_parquet(os.path.join(REPO, table))
    partner = None if code != 'Cp' else False   # no Cryptosporidium counterpart table is built
    rows.append(SA.evidence_of(code, nodes, layer_names(graph), partner=partner))
E = pd.DataFrame([vars(e) | {'edge_layers': len(e.edge_layers)} for e in rows])
E.to_csv(os.path.join(OUT, 'space_evidence.tsv'), sep='\t', index=False)
E

39 strategies registered


,code,genes,categorical,numeric,edge_layers,has_partner_table,has_orthogroups,has_literature_attention
0,Tg,8140,29,410,55,True,True,True
1,Pf,5720,25,141,28,True,True,True
2,Cp,3923,7,7,0,False,True,False


## 1 · The matrix

One row per strategy and space. A space with no graph loses the network strategies; a space with no counterpart table loses the transfer strategy; a space with no literature column loses the two that read attention.

In [2]:
frames = [SA.applicability(e) for e in rows]
A = pd.concat(frames, ignore_index=True)
A.to_csv(os.path.join(OUT, 'applicability.tsv'), sep='\t', index=False)
counts = A.groupby('space').applicable.agg(['sum', 'count'])
counts.columns = ['applicable', 'strategies']
counts['share'] = (100 * counts.applicable / counts.strategies).round(0)
json.dump({r.Index: int(r.applicable) for r in counts.itertuples()},
          open(os.path.join(OUT, 'summary.json'), 'w'), indent=1)
counts

,applicable,strategies,share
space,,,
Cp,27,39,69
Pf,39,39,100
Tg,39,39,100


## 2 · What each space cannot do, and why

The useful half of the answer: the missing requirement names what to build next for that space.

In [3]:
blocked = A[~A.applicable][['space', 'number', 'strategy', 'family', 'missing']]
blocked.to_csv(os.path.join(OUT, 'blocked.tsv'), sep='\t', index=False)
print(blocked.groupby(['space', 'missing']).size().to_string())
blocked

space  missing                                 
Cp     0 edge layers                               1
       no co-mention layer                         1
       no crosslink or IP-MS layer                 1
       no edge layer                               4
       no interaction layer                        1
       no literature-attention column              2
       no partner table declared for this space    1
       no structural-similarity layer              1


,space,number,strategy,family,missing
105,Cp,11,layer_propagation,Walk the networks,no edge layer
106,Cp,13,physical_partners,Walk the networks,no crosslink or IP-MS layer
107,Cp,14,structural_homology,Walk the networks,no structural-similarity layer
108,Cp,15,multiplex_modules,Walk the networks,0 edge layers
109,Cp,16,link_prediction,Walk the networks,no interaction layer
110,Cp,17,attention_correction,Walk the networks,no co-mention layer
111,Cp,18,unwritten_links,Walk the networks,no literature-attention column
112,Cp,29,ortholog_transfer,Cross species and strata,no partner table declared for this space
113,Cp,32,understudied_first,Combine strategies,no literature-attention column
114,Cp,33,neighbour_space,Combine strategies,no edge layer


## 3 · Reading it

For Cryptosporidium every missing requirement is an edge layer or a counterpart table, not a strategy that fails to apply to the organism: build its graph and its Toxoplasma counterpart table and most of the blocked strategies open. That is a build order, which is what WP8 was for.

In [4]:
cp = blocked[blocked.space == 'Cp']
needs_graph = cp[cp.missing.str.contains('layer')]
print(len(needs_graph), 'of', len(cp), 'blocked Cryptosporidium strategies wait on a graph')
print(sorted(set(cp.missing) - set(needs_graph.missing)))
cp.groupby('family').size().to_frame('blocked')

9 of 12 blocked Cryptosporidium strategies wait on a graph
['no literature-attention column', 'no partner table declared for this space']


,blocked
family,
Advanced models,1
Combine strategies,3
Cross species and strata,1
Walk the networks,7
